In [1]:
%pip install -q "uniface[cpu]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 6.0 MB/s eta 0:00:00


In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import uniface
from uniface.detection import RetinaFace
from uniface.gaze import MobileGaze
from uniface.draw import draw_gaze

print(f"UniFace version: {uniface.__version__}")

UniFace version: 4.0.0


In [3]:
# Initialize face detector
detector = RetinaFace(confidence_threshold=0.5)

# Initialize gaze estimator (uses ResNet34 by default)
gaze_estimator = MobileGaze()

Attempt 1/3: 100%|██████████| 11.9M/11.9M [00:00<00:00, 122MB/s]
Attempt 1/3: 100%|██████████| 81.5M/81.5M [00:00<00:00, 170MB/s]


# 1. Process All Test Images
Display original images in the first row and gaze-annotated images in the second row.

In [5]:
# Get all test images
demo_dir = Path('../assets/source')
demo_images = [demo_dir / n for n in ('gaze_away.jpg', 'gaze_averted.jpg', 'gaze_right.jpg',)]

# Store original and processed images
original_images = []
processed_images = []

for image_path in demo_images:
    print(f"Processing: {image_path.name}")

    # Load image
    image = cv2.imread(str(image_path))

    # Check if image was loaded successfully
    if image is None:
        print(f"  Error: Could not load image from {image_path}. Skipping.")
        continue

    original = image.copy()

    # Detect faces
    faces = detector.detect(image)
    print(f'  Detected {len(faces)} face(s)')

    # Estimate gaze for each face
    for i, face in enumerate(faces):
        x1, y1, x2, y2 = map(int, face.bbox[:4])
        face_crop = image[y1:y2, x1:x2]

        if face_crop.size > 0:
            gaze = gaze_estimator.estimate(face_crop)
            pitch_deg = np.degrees(gaze.pitch)
            yaw_deg = np.degrees(gaze.yaw)

            print(f'    Face {i+1}: pitch={pitch_deg:.1f}°, yaw={yaw_deg:.1f}°')

            # Draw gaze without angle text
            draw_gaze(image, face.bbox, gaze.pitch, gaze.yaw, draw_angles=False)

    # Convert BGR to RGB for display
    original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
    processed_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    original_images.append(original_rgb)
    processed_images.append(processed_rgb)

print(f"\nProcessed {len(demo_images)} images")

Processing: gaze_away.jpg
  Error: Could not load image from ../assets/source/gaze_away.jpg. Skipping.
Processing: gaze_averted.jpg
  Error: Could not load image from ../assets/source/gaze_averted.jpg. Skipping.
Processing: gaze_right.jpg
  Error: Could not load image from ../assets/source/gaze_right.jpg. Skipping.

Processed 3 images


# 2. Visualize Results
First row: Original images
Second row: Images with gaze direction arrows

In [7]:
num_images = len(original_images)

# Check if there are any images to display
if num_images == 0:
    print("No images were processed. Please ensure the image files exist at the specified path.")
else:
    # Create figure with 2 rows
    fig, axes = plt.subplots(2, num_images, figsize=(4*num_images, 8))

    # Handle case where there's only one image
    if num_images == 1:
        axes = axes.reshape(2, 1)

    # First row: Original images
    for i, img in enumerate(original_images):
        axes[0, i].imshow(img)
        axes[0, i].set_title(f'Original {i}', fontsize=12)
        axes[0, i].axis('off')

    # Second row: Gaze-annotated images
    for i, img in enumerate(processed_images):
        axes[1, i].imshow(img)
        axes[1, i].set_title(f'Gaze Estimation {i}', fontsize=12)
        axes[1, i].axis('off')

    plt.tight_layout()
    plt.show()

No images were processed. Please ensure the image files exist at the specified path.


Notes
pitch and yaw come back in radians, not degrees. Use np.degrees() before printing them.
MobileGaze() loads ResNet-34. Pass model_name= for one of the other backbones.
# Gaze is where the eyes point, which is not the same as where the head points. Notebook 11 covers the latter